<a href="https://colab.research.google.com/github/taniavalasky/TCC_Otimizacao_de_Carteira_de_Acoes/blob/main/simple_case.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 57.6 MB/s  0:00:00


In [4]:
import gurobipy as gp
from gurobipy import GRB
import yfinance as yf
import pandas as pd
import pandas_datareader.data as pdr
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import math

## Dados

In [5]:
# Simple case:
# Covariance matrix
Q = np.array([[0.038, 0.02,  0.017, 0.014, 0.019, 0.017],
              [0.02,  0.043, 0.015, 0.013, 0.021, 0.014],
              [0.017, 0.015, 0.034, 0.011, 0.014, 0.014],
              [0.014, 0.013, 0.011, 0.044, 0.014, 0.011],
              [0.019, 0.021, 0.014, 0.014, 0.04,  0.014],
              [0.017, 0.014, 0.014, 0.011, 0.014, 0.046]])

# Expected returns
v = np.array([0.021, 0.04, -0.034, -0.028, -0.005, 0.006])

# Number of assets
n = Q.shape[1]  # ou Q.shape[0], já que é quadrada


## Modelo

In [6]:
# Q: matriz de covariância (n x n)
# v: vetor de retornos médios (n,)
# rho: retorno mínimo exigido

rho = 0.0003   # retorno mínimo
alpha = 6    # número máximo de ativos

model = gp.Model("Modelo Exato")

# Variáveis
x = model.addMVar(n, vtype=GRB.CONTINUOUS, name="x")
y = model.addMVar(n, lb=0, ub=1, vtype=GRB.BINARY, name="y")

# Função objetivo (minimizar risco)
obj = x @ Q @ x
model.setObjective(obj, GRB.MINIMIZE)

# Restrições
model.addConstr(v @ x >= rho, "retorno_minimo")    # retorno mínimo
model.addConstr(x.sum() == 1, "vetor de uns")   # soma dos pesos = 1
model.addConstr(x <= y, "link_x_y")  # se y_i = 0, então x_i = 0
model.addConstr(y.sum() <= alpha, "somatorio")  # a soma dos ativos escolhidos não pode ultrapassar a quantidade maxima de ativos

# Otimização
model.optimize()


Restricted license - for non-production use only - expires 2028-11-29
Gurobi Optimizer version 14.0.0 build v14.0.0rc1 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 9 rows, 12 columns and 30 nonzeros (Min)
Model fingerprint: 0xbf410775
Model has 0 linear objective coefficients
Model has 21 quadratic objective terms
Variable types: 6 continuous, 6 integer (6 binary)
Coefficient statistics:
  Matrix range     [5e-03, 1e+00]
  Objective range  [0e+00, 0e+00]
  QObjective range [4e-02, 9e-02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e-04, 6e+00]

Presolve removed 7 rows and 6 columns
Presolve time: 0.00s
Presolved: 2 rows, 6 columns, 11 nonzeros
Presolved model has 21 quadratic objective terms
Variable types: 6 continuous, 0 integer (0 binary)
Presolve time: 0.02s
Presolved: 2 rows, 6 columns, 11 nonzeros
Presolved m

## Risco e Retorno

In [7]:
risco = np.sqrt(x.X @ Q @ x.X)  # x.X é o vetor de pesos ótimo retornado pelo Gurobi
retorno = v @ x.X

print("O risco da carteira é: ", risco)
print("O retorno da carteira é: ", retorno)

O risco da carteira é:  0.13934154657934836
O retorno da carteira é:  0.00030000047336986666


In [8]:
print(f"O risco da carteira é: {risco:.4f}")
print(f"O retorno da carteira é: {retorno:.4f}")

O risco da carteira é: 0.1393
O retorno da carteira é: 0.0003
